In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# --- WHITE THEME ---
# ==========================================
BLUE = "#1f77b4"
ORANGE = "#ff7f0e"
RED = "#d62728"
GREEN = "#2ca02c"
GREY = "rgba(120,120,120,0.55)"
BLACK_LINE = "rgba(0,0,0,0.45)"
GRID = "rgba(0,0,0,0.15)"


def apply_white_theme(fig, title):
    fig.update_layout(
        height=650,
        width=1000,
        title_text=title,
        plot_bgcolor="white",
        paper_bgcolor="white",
        font=dict(color="black"),
        showlegend=False,
        margin=dict(b=100, t=80, l=50, r=50),
    )

    fig.update_xaxes(
        showgrid=True,
        gridcolor=GRID,
        linecolor="black",
        tickfont=dict(color="black"),
        title_font=dict(color="black"),
        zeroline=True,
        zerolinecolor="rgba(0,0,0,0.25)",
    )

    fig.update_yaxes(
        showgrid=True,
        gridcolor=GRID,
        linecolor="black",
        tickfont=dict(color="black"),
        title_font=dict(color="black"),
        zeroline=True,
        zerolinecolor="rgba(0,0,0,0.25)",
    )

    return fig


# ==========================================
# --- SIMULATION PARAMETERS ---
# ==========================================
n_samples = 200
n_steps_p2 = 100
n_steps_p3 = 100
total_trades = n_steps_p2 + n_steps_p3

true_val_d6 = 3.5
true_val_d10 = 5.5
mm_spread = 0.4
decay_rate = 4.5

np.random.seed(42)

# ==========================================
# --- ENVIRONMENT GENERATION ---
# ==========================================
all_prices = np.round(np.arange(1.0, 11.1, 0.1), 1)

vols_d6 = np.exp(-decay_rate * np.abs(all_prices - true_val_d6))
probs_d6 = vols_d6 / np.sum(vols_d6)

vols_d10 = np.exp(-decay_rate * np.abs(all_prices - true_val_d10))
probs_d10 = vols_d10 / np.sum(vols_d10)

theo_x_d6 = np.arange(1, 7)
theo_y_d6 = np.ones(6) / 6.0

theo_x_d10 = np.arange(1, 11)
theo_y_d10 = np.ones(10) / 10.0

colors_d6 = []
colors_d10 = []

for p in all_prices:
    colors_d6.append(
        "rgba(31,119,180,0.65)"
        if p < true_val_d6
        else "rgba(255,127,14,0.65)"
        if p > true_val_d6
        else GREY
    )

    colors_d10.append(
        "rgba(31,119,180,0.65)"
        if p < true_val_d10
        else "rgba(255,127,14,0.65)"
        if p > true_val_d10
        else GREY
    )

# ==========================================
# --- PHASE 1: DATA GENERATION ---
# ==========================================
est_rolls = np.random.randint(1, 7, size=n_samples)
rolling_means = np.cumsum(est_rolls) / np.arange(1, n_samples + 1)

final_estimated_val = rolling_means[-1]
mm_half_spread = mm_spread / 2.0

mm_bid = final_estimated_val - mm_half_spread
mm_ask = final_estimated_val + mm_half_spread

# ==========================================
# --- PHASE 2 & 3: TRADING SIMULATION ---
# ==========================================
trad_rolls = np.concatenate(
    [
        np.random.randint(1, 7, size=n_steps_p2),
        np.random.randint(1, 11, size=n_steps_p3),
    ]
)

traded_arrival_prices = []
profits = []

for t in range(total_trades):
    current_probs = probs_d6 if t < n_steps_p2 else probs_d10
    trade_executed = False

    while not trade_executed:
        p = np.random.choice(all_prices, p=current_probs)

        if p <= mm_bid:
            execution_price = mm_bid
            is_buy = True
            trade_executed = True

        elif p >= mm_ask:
            execution_price = mm_ask
            is_buy = False
            trade_executed = True

    settlement_val = trad_rolls[t]

    profit = (
        settlement_val - execution_price
        if is_buy
        else execution_price - settlement_val
    )

    traded_arrival_prices.append(p)
    profits.append(profit)

cum_profits = np.insert(np.cumsum(profits), 0, 0)
time_grid_trad = np.arange(total_trades + 1)
plot_trad_rolls = np.insert(trad_rolls, 0, final_estimated_val)

min_profit = min(-20, np.min(cum_profits) * 1.1)
max_profit = max(20, np.max(cum_profits) * 1.1)

# ==========================================
# --- ANIMATION FRAMES ---
# ==========================================
frames = []
total_frames = n_samples + total_trades + 1
hist_bins = np.arange(0.5, 11.5, 1)

for k in range(total_frames):

    t_trad = max(0, k - n_samples)
    current_x_range = [0, max(10, t_trad + 5)]

    # ======================================
    # PHASE 1: Estimation period
    # ======================================
    if k <= n_samples:

        current_est_rolls = est_rolls[:k]
        current_mean = rolling_means[k - 1] if k > 0 else true_val_d6

        if k > 0:
            hist_counts, _ = np.histogram(
                current_est_rolls,
                bins=hist_bins,
                density=True
            )
        else:
            hist_counts = np.zeros(len(hist_bins) - 1)

        book_vols = vols_d6
        book_colors = list(colors_d6)

        true_val_line = true_val_d6
        theo_x = theo_x_d6
        theo_y = theo_y_d6
        theo_color = "rgba(31,119,180,0.25)"
        theo_line = BLUE

        x_d6, y_d6 = [np.nan], [np.nan]
        x_d10, y_d10 = [np.nan], [np.nan]
        t_x_eq, t_y_eq = [np.nan], [np.nan]

        plot_mm_bid = current_mean
        plot_mm_ask = current_mean

        frame_name = f"est{k}"

    # ======================================
    # PHASE 2 & 3: Trading period
    # ======================================
    else:

        current_mean = final_estimated_val
        plot_mm_bid = mm_bid
        plot_mm_ask = mm_ask

        hist_counts, _ = np.histogram(
            est_rolls,
            bins=hist_bins,
            density=True
        )

        # ----------------------------------
        # Phase 2: D6 regime
        # ----------------------------------
        if t_trad <= n_steps_p2:

            book_vols = vols_d6
            book_colors = list(colors_d6)

            true_val_line = true_val_d6
            theo_x = theo_x_d6
            theo_y = theo_y_d6
            theo_color = "rgba(31,119,180,0.25)"
            theo_line = BLUE

            x_d6 = time_grid_trad[:t_trad + 1]
            y_d6 = plot_trad_rolls[:t_trad + 1]

            x_d10, y_d10 = [np.nan], [np.nan]

        # ----------------------------------
        # Phase 3: D10 regime
        # ----------------------------------
        else:

            book_vols = vols_d10
            book_colors = list(colors_d10)

            true_val_line = true_val_d10
            theo_x = theo_x_d10
            theo_y = theo_y_d10
            theo_color = "rgba(255,127,14,0.25)"
            theo_line = ORANGE

            x_d6 = time_grid_trad[:n_steps_p2 + 1]
            y_d6 = plot_trad_rolls[:n_steps_p2 + 1]

            x_d10 = time_grid_trad[n_steps_p2:t_trad + 1]
            y_d10 = plot_trad_rolls[n_steps_p2:t_trad + 1]

        # Highlight the latest market-arrival price
        if t_trad > 0:
            p_t = traded_arrival_prices[t_trad - 1]
            idx = np.where(np.isclose(all_prices, p_t))[0][0]

            book_colors[idx] = BLUE if p_t <= mm_bid else RED

        t_x_eq = time_grid_trad[:t_trad + 1]
        t_y_eq = cum_profits[:t_trad + 1]

        frame_name = f"trade{t_trad}"

    # ======================================
    # TRACES
    # ======================================
    tr0 = go.Bar(
        x=np.arange(1, 12),
        y=hist_counts,
        marker=dict(color="rgba(31,119,180,0.55)")
    )

    tr1 = go.Bar(
        x=theo_x,
        y=theo_y,
        marker=dict(
            color=theo_color,
            line=dict(color=theo_line, width=2)
        )
    )

    tr2 = go.Scatter(
        x=[current_mean, current_mean],
        y=[0, 0.25],
        mode="lines",
        line=dict(color=RED, width=3)
    )

    tr3 = go.Bar(
        x=all_prices,
        y=book_vols,
        marker=dict(color=book_colors)
    )

    tr4 = go.Scatter(
        x=[true_val_line, true_val_line],
        y=[0, 1],
        mode="lines",
        line=dict(color=BLACK_LINE, width=2, dash="dash")
    )

    tr5 = go.Scatter(
        x=[current_mean, current_mean],
        y=[0, 1],
        mode="lines",
        line=dict(color=RED, width=3, dash="dash")
    )

    tr6 = go.Scatter(
        x=[plot_mm_bid, plot_mm_bid],
        y=[0, 1],
        mode="lines",
        line=dict(color=RED, width=2)
    )

    tr7 = go.Scatter(
        x=[plot_mm_ask, plot_mm_ask],
        y=[0, 1],
        mode="lines",
        line=dict(color=RED, width=2)
    )

    tr8 = go.Scatter(
        x=x_d6,
        y=y_d6,
        mode="lines+markers",
        line=dict(color=BLUE, width=2, shape="hv"),
        marker=dict(size=4)
    )

    tr9 = go.Scatter(
        x=x_d10,
        y=y_d10,
        mode="lines+markers",
        line=dict(color=ORANGE, width=2, shape="hv"),
        marker=dict(size=4)
    )

    tr10 = go.Scatter(
        x=t_x_eq,
        y=t_y_eq,
        mode="lines",
        line=dict(color=GREEN, width=3, shape="hv")
    )

    frames.append(
        go.Frame(
            data=[
                tr0, tr1, tr2,
                tr3, tr4, tr5, tr6, tr7,
                tr8, tr9, tr10
            ],
            name=frame_name,
            layout=go.Layout(
                xaxis2=dict(range=current_x_range),
                xaxis4=dict(range=current_x_range)
            )
        )
    )
#========================================== # --- FIGURE INITIALIZATION --- # ========================================== 
fig = make_subplots( rows=2, cols=2, subplot_titles=[ "Phase 1: True Distribution vs LLN Sampling", "Phase 2 & 3: Settlement Volatility", "Market Arrivals vs Frozen MM Quotes", "Phase 2 & 3: MM Equity Curve" ], horizontal_spacing=0.10, vertical_spacing=0.15 ) 
fig.add_trace(go.Bar(x=[np.nan], y=[np.nan]), row=1, col=1) 
fig.add_trace(go.Bar(x=[np.nan], y=[np.nan]), row=1, col=1) 
fig.add_trace(go.Scatter(x=[np.nan], y=[np.nan]), row=1, col=1) 
fig.add_trace(go.Bar(x=[np.nan], y=[np.nan]), row=2, col=1) 
fig.add_trace(go.Scatter(x=[np.nan], y=[np.nan]), row=2, col=1) 
fig.add_trace(go.Scatter(x=[np.nan], y=[np.nan]), row=2, col=1) 
fig.add_trace(go.Scatter(x=[np.nan], y=[np.nan]), row=2, col=1) 
fig.add_trace(go.Scatter(x=[np.nan], y=[np.nan]), row=2, col=1) 
fig.add_trace(go.Scatter(x=[np.nan], y=[np.nan]), row=1, col=2) 
fig.add_trace(go.Scatter(x=[np.nan], y=[np.nan]), row=1, col=2) 
fig.add_trace(go.Scatter(x=[np.nan], y=[np.nan]), row=2, col=2) 
fig.frames = frames 

# ==========================================
# --- SLIDER & LAYOUT ---
# ==========================================
sliders = [
    dict(
        active=0,
        currentvalue={"prefix": "Step: "},
        pad={"t": 0},
        x=0.15,
        len=0.85,
        y=-0.1,
        steps=[
            dict(
                method="animate",
                args=[
                    [frames[k].name],
                    dict(
                        mode="immediate",
                        frame=dict(duration=0, redraw=True),
                        transition=dict(duration=0),
                    ),
                ],
                label=f"Est {k}" if k <= n_samples else f"Trade {k - n_samples}",
            )
            for k in range(total_frames)
        ],
    )
]

fig.update_layout(
    sliders=sliders,
    updatemenus=[
        dict(
            type="buttons",
            x=0.0,
            y=-0.15,
            xanchor="left",
            yanchor="top",
            direction="left",
            showactive=False,
            buttons=[
                dict(
                    label="▶ Play",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(duration=40, redraw=True),
                            transition=dict(duration=0),
                            fromcurrent=True,
                            mode="immediate",
                        ),
                    ],
                )
            ],
        )
    ],
)

apply_white_theme(
    fig,
    "Regime Shift: The Danger of Non-Stationarity in Market Making",
)

# ==========================================
# --- AXES STYLING ---
# ==========================================
fig.update_xaxes(
    title_text="Dice Roll",
    range=[0.5, 11.5],
    dtick=1,
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text="Probability",
    range=[0, 0.25],
    row=1,
    col=1,
)

fig.update_xaxes(
    title_text="Price",
    range=[0.5, 11.5],
    dtick=1,
    row=2,
    col=1,
)

fig.update_yaxes(
    title_text="Volume Mass",
    range=[0, 1.1],
    row=2,
    col=1,
)

fig.update_xaxes(
    title_text="Number of Trades",
    range=[0, 10],
    row=1,
    col=2,
)

fig.update_yaxes(
    title_text="Settlement (1-10)",
    range=[0.5, 11.5],
    dtick=1,
    row=1,
    col=2,
)

fig.update_xaxes(
    title_text="Number of Trades",
    range=[0, 10],
    row=2,
    col=2,
)

fig.update_yaxes(
    title_text="Cumulative Profit",
    range=[min_profit, max_profit],
    row=2,
    col=2,
)


# ==========================================
# --- SLIDER & LAYOUT ---
# ==========================================
sliders = [
    dict(
        active=0,
        currentvalue={"prefix": "Step: "},
        pad={"t": 0},
        x=0.15,
        len=0.85,
        y=-0.1,
        steps=[
            dict(
                method="animate",
                args=[
                    [frames[k].name],
                    dict(
                        mode="immediate",
                        frame=dict(duration=0, redraw=True),
                        transition=dict(duration=0),
                    ),
                ],
                label=f"Est {k}" if k <= n_samples else f"Trade {k - n_samples}",
            )
            for k in range(total_frames)
        ],
    )
]

fig.update_layout(
    sliders=sliders,
    updatemenus=[
        dict(
            type="buttons",
            x=0.0,
            y=-0.15,
            xanchor="left",
            yanchor="top",
            direction="left",
            showactive=False,
            buttons=[
                dict(
                    label="▶ Play",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(duration=40, redraw=True),
                            transition=dict(duration=0),
                            fromcurrent=True,
                            mode="immediate",
                        ),
                    ],
                )
            ],
        )
    ],
)

apply_white_theme(
    fig,
    "Regime Shift: The Danger of Non-Stationarity in Market Making",
)

# ==========================================
# --- AXES STYLING ---
# ==========================================
fig.update_xaxes(
    title_text="Dice Roll",
    range=[0.5, 11.5],
    dtick=1,
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text="Probability",
    range=[0, 0.25],
    row=1,
    col=1,
)

fig.update_xaxes(
    title_text="Price",
    range=[0.5, 11.5],
    dtick=1,
    row=2,
    col=1,
)

fig.update_yaxes(
    title_text="Volume Mass",
    range=[0, 1.1],
    row=2,
    col=1,
)

fig.update_xaxes(
    title_text="Number of Trades",
    range=[0, 10],
    row=1,
    col=2,
)

fig.update_yaxes(
    title_text="Settlement (1-10)",
    range=[0.5, 11.5],
    dtick=1,
    row=1,
    col=2,
)

fig.update_xaxes(
    title_text="Number of Trades",
    range=[0, 10],
    row=2,
    col=2,
)

fig.update_yaxes(
    title_text="Cumulative Profit",
    range=[min_profit, max_profit],
    row=2,
    col=2,
)

fig.show()